# Methode B - LLM-based Transformation (Metadata)

**Concept:** The LLM receives the parsed Figma JSON and PrimeVue documentation as context and directly generates Vue 3 single-file components (SFCs) — without any explicit rules.

**Models:**

| Display name                    | OpenRouter model ID             | Provider   |
|---------------------------------|---------------------------------|------------|
| `anthropic/claude-sonnet-5`     | `anthropic/claude-sonnet-5`     | Anthropic  |
| `openai/gpt-5.6-terra`          | `openai/gpt-5.6-terra`          | OpenAI     |
| `google/gemini-3.1-pro-preview` | `google/gemini-3.1-pro-preview` | Google     |
| `moonshotai/kimi-k3`            | `moonshotai/kimi-k3`            | MoonshotAI |

**Three Context Strategies (Variants):**

| Variant | Context                                     | Character                                       |
|---------|---------------------------------------------|-------------------------------------------------|
| **B1**  | No documentation                            | minimal, no api reference, error-prone          |
| **B2**  | Docs only for detected components (raw)     | focused, relevant, but noisy                    |
| **B3**  | Docs only for detected components (cleaned) | focused, relevant, clean, best expected results |

=> Variant with all raw docs not tested, because the large context would exceed the token limit for many files. The cleaned docs (B3) are expected to perform best, but B2 is also interesting to see the effect of cleaning.

**Prompt strategy:** Zero-shot and few-shot examples, no chain-of-thought.


In [104]:
import os
import re
import csv
import json
import time
import urllib.request
from pathlib import Path
from dotenv import load_dotenv

In [105]:
INPUT_DIR = 'dataset/figma-data/cleaned/components'
OUTPUT_DIR = 'dataset/storybook/src/stories/components'

DOCS_DIR_RAW = 'primevue/component-documentation/raw'
DOCS_DIR_CLEANED = 'primevue/component-documentation/cleaned'

# 'ZERO-SHOT' or 'FEW-SHOT'
PROMPT_STRATEGY = 'ZERO-SHOT'

API_URL = 'https://openrouter.ai/api/v1/chat/completions'

API_MODELS = {
    'claude-sonnet-5': 'anthropic/claude-sonnet-5', # 2 / 10
    'gpt-5.6-terra': 'openai/gpt-5.6-terra', # 2,50 / 15
    'gemini-3.1-pro': 'google/gemini-3.1-pro-preview', # 2 / 12
    'kimi-k2.6': 'moonshotai/kimi-k2.6', # 0,6 / 3,41
}

API_TEMPERATURE=0.0               # 0.0–2.0 (1.0 = default; lower = more deterministic)
API_REASONING_EFFORT='medium'   # 'low' | 'medium' | 'high'

load_dotenv(dotenv_path=Path('.env'))

OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

if OPENROUTER_API_KEY is None:
    raise ValueError('OPENROUTER_API_KEY not found in environment variables. Please set it in the .env file.')

## Load data and documentations

### 1.1 Load Raw Component-Documentations

In [106]:
DOCS_DIR_PATH = Path(DOCS_DIR_RAW)
RAW_DOCS: dict[str, str] = {}

for md_file in sorted(DOCS_DIR_PATH.glob('*.md')):
    RAW_DOCS[md_file.stem.lower()] = md_file.read_text(encoding='utf-8', errors='ignore')

print(f'Loaded Docs: {len(RAW_DOCS)}')
for name, content in RAW_DOCS.items():
    tokens_est = len(content) // 4
    print(f'  {name:20s}  ~{tokens_est:5d} Tokens')

TOTAL_TOKENS_B2 = sum(len(c) // 4 for c in RAW_DOCS.values())
print(f'\nTotal B2-Context: ~{TOTAL_TOKENS_B2:,} Tokens')

Loaded Docs: 25
  accordion             ~ 8091 Tokens
  avatar                ~ 2737 Tokens
  badge                 ~ 1860 Tokens
  breadcrumb            ~  988 Tokens
  button                ~14672 Tokens
  card                  ~ 1426 Tokens
  checkbox              ~ 4414 Tokens
  datatable             ~20664 Tokens
  datepicker            ~11753 Tokens
  dialog                ~ 9667 Tokens
  divider               ~ 4877 Tokens
  inputnumber           ~ 7011 Tokens
  inputtext             ~ 6664 Tokens
  menu                  ~ 3240 Tokens
  password              ~ 8082 Tokens
  popover               ~ 3859 Tokens
  progressbar           ~ 1173 Tokens
  radiobutton           ~ 4283 Tokens
  select                ~11278 Tokens
  skeleton              ~ 3134 Tokens
  slider                ~ 2688 Tokens
  tabs                  ~ 7255 Tokens
  tag                   ~ 1966 Tokens
  textarea              ~ 6287 Tokens
  toggleswitch          ~ 2693 Tokens

Total B2-Context: ~150,762 Tokens

### 1.2 Load Cleand Component-Documentations

In [107]:
DOCS_DIR_PATH = Path(DOCS_DIR_CLEANED)
CLEANED_DOCS: dict[str, str] = {}

for md_file in sorted(DOCS_DIR_PATH.glob('*.md')):
    CLEANED_DOCS[md_file.stem.lower()] = md_file.read_text(encoding='utf-8', errors='ignore')

print(f'Loaded Docs: {len(CLEANED_DOCS)}')
for name, content in CLEANED_DOCS.items():
    tokens_est = len(content) // 4
    print(f'  {name:20s}  ~{tokens_est:5d} Tokens')

TOTAL_TOKENS_B3 = sum(len(c) // 4 for c in CLEANED_DOCS.values())
print(f'\nTotal B3-Context: ~{TOTAL_TOKENS_B3:,} Tokens')

Loaded Docs: 25
  accordion             ~ 1811 Tokens
  avatar                ~  432 Tokens
  badge                 ~  283 Tokens
  breadcrumb            ~  169 Tokens
  button                ~ 3820 Tokens
  card                  ~  501 Tokens
  checkbox              ~  626 Tokens
  datatable             ~ 2480 Tokens
  datepicker            ~ 1755 Tokens
  dialog                ~ 1349 Tokens
  divider               ~  725 Tokens
  inputnumber           ~ 1634 Tokens
  inputtext             ~ 3649 Tokens
  menu                  ~  539 Tokens
  password              ~ 4501 Tokens
  popover               ~  761 Tokens
  progressbar           ~  272 Tokens
  radiobutton           ~  571 Tokens
  select                ~ 2065 Tokens
  skeleton              ~  438 Tokens
  slider                ~  487 Tokens
  tabs                  ~ 1641 Tokens
  tag                   ~  360 Tokens
  textarea              ~ 3549 Tokens
  toggleswitch          ~  587 Tokens

Total B3-Context: ~35,005 Tokens


### 1.3 Load Figma JSON Data

In [108]:
INPUT_DIR_PATH = Path(INPUT_DIR)
FIGMA_DATA: dict[str, dict] = {}

input_files = sorted(
    f for f in INPUT_DIR_PATH.rglob('*.json')
    if f.parent != INPUT_DIR_PATH
)

# For testing: only load one file per complexity level
#seen_complexities = set()
#input_files = []
#for f in all_input_files:
#    complexity = f.parent.name
#
#    if complexity not in seen_complexities:
#        seen_complexities.add(complexity)
#
#        input_files.append(f)

for input_file in input_files:
    print(f'Loading {input_file}...')

    with open(input_file, encoding='utf-8-sig') as f:
        key = f'{input_file.parent.name}-{input_file.stem}'.lower()
        FIGMA_DATA[key] = json.load(f)

print(f'\nLoaded Figma JSONs: {len(FIGMA_DATA)}')
for name, data in FIGMA_DATA.items():
    tokens_est = len(json.dumps(data)) // 4

    print(f'  {name:20s}  ~{tokens_est:5d} Tokens')

Loading dataset\figma-data\cleaned\components\hard\1.json...
Loading dataset\figma-data\cleaned\components\hard\10.json...
Loading dataset\figma-data\cleaned\components\hard\2.json...
Loading dataset\figma-data\cleaned\components\hard\3.json...
Loading dataset\figma-data\cleaned\components\hard\4.json...
Loading dataset\figma-data\cleaned\components\hard\5.json...
Loading dataset\figma-data\cleaned\components\hard\6.json...
Loading dataset\figma-data\cleaned\components\hard\7.json...
Loading dataset\figma-data\cleaned\components\hard\8.json...
Loading dataset\figma-data\cleaned\components\hard\9.json...
Loading dataset\figma-data\cleaned\components\medium\1.json...
Loading dataset\figma-data\cleaned\components\medium\10.json...
Loading dataset\figma-data\cleaned\components\medium\2.json...
Loading dataset\figma-data\cleaned\components\medium\3.json...
Loading dataset\figma-data\cleaned\components\medium\4.json...
Loading dataset\figma-data\cleaned\components\medium\5.json...
Loading da

## 2. Detect PrimeVue-Components in Figma Data

In [109]:
KNOWN_COMPONENTS = set(CLEANED_DOCS.keys())

# Compound components that appear as FRAME
FRAME_COMPONENTS = {
    'card', 'dialog', 'tabs', 'datatable', 'select',
    'popover', 'breadcrumb', 'accordion',
}
# Aliases (Figma-Name → Doc-Name)
DOC_ALIASES = {
    'calendar': 'datepicker',
    'overlaybadge': 'badge',
}


def _normalize(name: str) -> str:
    return re.sub(r'[\s\-_]+', '', name or '').lower()


def detect_components(figma_node: dict, found: set | None = None) -> set[str]:
    """Collect all PrimeVue component names that appear in the Figma JSON"""
    if found is None:
        found = set()

    if not isinstance(figma_node, dict):
        return found

    name = figma_node.get('name', '')
    t = figma_node.get('type', '')

    if name.startswith('_'):
        return found  # interne Sub-Instance

    norm = _normalize(name)
    norm = DOC_ALIASES.get(norm, norm)

    if t in ('INSTANCE', 'FRAME') and norm in KNOWN_COMPONENTS:
        found.add(norm)

    for child in figma_node.get('children', []) or []:
        detect_components(child, found)

    return found


DETECTED_COMPONENTS: dict[str, set[str]] = {}

for key, data in FIGMA_DATA.items():
    detected = detect_components(data)
    DETECTED_COMPONENTS[key] = detected

    print(f'{key:20s}  → Detected: {", ".join(sorted(detected)) or "None"}')

print(f'\nTotal Unique Detected Components: {len(set.union(*DETECTED_COMPONENTS.values()))}')

hard-1                → Detected: button, checkbox, dialog, inputtext, select
hard-10               → Detected: button, datatable, popover
hard-2                → Detected: datatable
hard-3                → Detected: button, datepicker, inputnumber, inputtext, select
hard-4                → Detected: button, datatable, popover
hard-5                → Detected: button, dialog, tabs
hard-6                → Detected: inputtext, select
hard-7                → Detected: avatar, button, card, divider, tag
hard-8                → Detected: button, card, tabs
hard-9                → Detected: avatar, breadcrumb, button, datepicker, dialog, divider, inputtext, textarea
medium-1              → Detected: button, card, checkbox, inputtext, password
medium-10             → Detected: button, card, progressbar
medium-2              → Detected: tabs
medium-3              → Detected: accordion
medium-4              → Detected: breadcrumb, button, menu
medium-5              → Detected: button, card, inp

## 3. Prompt Building

### 3.1 Context-Builder

In [110]:
def build_context(strategy: str, figma_node: dict | None = None) -> tuple[str, list[str]]:
    """Returns (context_string, used_components)

    strategy: 'b1' | 'b2' | 'b3'
    figma_node: Required b2 and b3 for component recognition
    """

    if strategy == 'b1':
        return '', []  # No context for B1

    if figma_node is None:
        raise ValueError('Strategies requires figma_node for component recognition.')

    docs = RAW_DOCS if strategy == 'b2' else CLEANED_DOCS

    detected_components = detect_components(figma_node)

    context_parts = []
    used_components = []

    for comp in sorted(detected_components):
        if comp in docs:
            print(f'  Adding doc for component: {comp} ({len(docs[comp])//4} tokens)')

            context_parts.append(f'# {comp}\n\n{docs[comp]}')
            used_components.append(comp)

    return '\n\n'.join(context_parts), used_components # Only docs of detected components in raw or cleaned form, depending on strategy

### 3.2 Prompt Templates

In [111]:
SYSTEM_INSTRUCTIONS = """You are an expert Vue 3 and PrimeVue developer.
Analyse the given Figma mockup JSON data and transform it into a complete, working Vue 3 Single File Component with PrimeVue 4 components. Use the provided PrimeVue documentation if given as reference for component usage and props.

STRICT REQUIREMENTS:
- Use PrimeVue 4 components exclusively for UI elements
- Use <script setup> syntax (no Options API)
- Import every PrimeVue component used: import Button from 'primevue/button'
- Use Tailwind CSS utility classes for layout and spacing
- Use ref() from Vue for all form/input state
- Map Figma Auto-Layout (HORIZONTAL/VERTICAL) to flex/flex-col
- Map itemSpacing to gap-*, padding values to p-*/px-*/py-*
- Output ONLY the Vue SFC — no explanation, no markdown fences, no prose
- Return exactly one complete Vue SFC, starting directly with <template> and ending with </script>
- If required details are missing or ambiguous, do not invent unsupported behavior; use the closest valid structural mapping supported by the Figma JSON
- Treat the transformation as incomplete until all eligible non-ignored nodes are represented in the output
- Before finalizing, verify that the SFC is syntactically valid, all used PrimeVue components are imported, and all form/input state uses ref()
- Assume PrimeVue Aura theme as baseline for styling; do not generate custom theme CSS unless explicitly required by Figma mockup JSON data

FIGMA JSON DATA STRUCTURE:
- type=INSTANCE, name=<component>: a PrimeVue component instance
- componentProperties: Figma design properties to map to PrimeVue props
- type=FRAME: layout container → <div> with Tailwind classes
- type=TEXT: standalone text → <span> or semantic element
- Nodes with name starting with '_' are internal sub-instances (ignore them)"""

FEW_SHOT_EXAMPLES = '''FEW-SHOT EXAMPLES for mapping Figma JSON to Vue 3 SFCs:

Example SIMPLE composite:
```json
{
  "type": "FRAME",
  "name": "Column [Simple Composite]",
  "layoutMode": "VERTICAL",
  "itemSpacing": 16.0,
  "paddingLeft": 24.0,
  "paddingRight": 24.0,
  "paddingTop": 24.0,
  "paddingBottom": 24.0,
  "children": [
    {
      "type": "FRAME",
      "name": "Row",
      "layoutMode": "HORIZONTAL",
      "itemSpacing": 16.0,
      "children": [
        {
          "type": "INSTANCE",
          "name": "avatar",
          "componentProperties": {
            "Text": "B",
            "Show Badge": false,
            "Size": "X-Large",
            "Type": "Label",
            "Circle": "True"
          }
        },
        {
          "type": "TEXT",
          "name": "Benutzername",
          "characters": "Benutzername"
        }
      ]
    },
    {
      "type": "INSTANCE",
      "name": "textarea",
      "componentProperties": {
        "Float Label": "Placeholder",
        "Show Text": true,
        "State": "Default",
        "Invalid": "False",
        "Disabled": "False",
        "Filled": "False",
        "Size": "Normal",
        "Ifta Label": "False",
        "Float Label": "False",
        "Float Label Variant": "N/A"
      }
    },
    {
      "type": "FRAME",
      "name": "Row",
      "layoutMode": "HORIZONTAL",
      "children": [
        {
          "type": "INSTANCE",
          "name": "checkbox",
          "componentProperties": {
            "Label": "Benachrichtigen",
            "Show Label": true,
            "Hover": "False",
            "Selected": "False",
            "Focus": "False",
            "Disabled": "False",
            "Filled": "False",
            "Size": "Normal"
          }
        },
        {
          "type": "INSTANCE",
          "name": "button",
          "componentProperties": {
            "Left Icon": "7:2160",
            "Icon": "7:2160",
            "Text": "Senden",
            "Show Right Icon": false,
            "Right Icon": "7:2160",
            "Show Left Icon": false,
            "Severity": "Primary",
            "State": "Idle",
            "Disabled": "False",
            "Icon Only": "False",
            "Raised": "False",
            "Rounded": "False",
            "Text": "False",
            "Outlined": "False",
            "Link": "False"
          }
        }
      ]
    }
  ]
}
```

```vue
<template>
  <div class="flex w-lg flex-col gap-4 p-6">
    <div class="flex items-center gap-4">
      <Avatar label="B" size="xlarge" shape="circle" />
      <span class="text-xl text-black">Benutzername</span>
    </div>
    <Textarea v-model="feedback" placeholder="Feedback eingeben..." />
    <div class="flex items-center justify-between">
      <div class="flex items-center gap-2">
        <Checkbox v-model="notification" input-id="notification" binary />
        <label for="notification">Benachrichtigen</label>
      </div>
      <Button label="Senden" severity="primary" class="w-fit" />
    </div>
  </div>
</template>

<script setup lang="ts">
  import { ref } from 'vue'
  import Avatar from 'primevue/avatar'
  import Button from 'primevue/button'
  import Textarea from 'primevue/textarea'
  import Checkbox from 'primevue/checkbox'

  const feedback = ref('')
  const notification = ref(false)
</script>
```

Example MEDIUM composite:
```json
{
  "type": "FRAME",
  "name": "Card [Medium Composite]",
  "layoutMode": "VERTICAL",
  "itemSpacing": 7.0,
  "children": [
    {
      "type": "FRAME",
      "name": "body",
      "layoutMode": "VERTICAL",
      "itemSpacing": 16.0,
      "paddingLeft": 24.0,
      "paddingRight": 24.0,
      "paddingTop": 24.0,
      "paddingBottom": 24.0,
      "children": [
        {
          "type": "FRAME",
          "name": "caption",
          "layoutMode": "VERTICAL",
          "itemSpacing": 7.0,
          "children": [
            {
              "type": "TEXT",
              "name": "Anmelden",
              "characters": "Anmelden"
            }
          ]
        },
        {
          "type": "FRAME",
          "name": "content",
          "layoutMode": "VERTICAL",
          "itemSpacing": 16.0,
          "children": [
            {
              "type": "INSTANCE",
              "name": "inputtext",
              "componentProperties": {
                "Float Label": "Placeholder",
                "Show Label": false,
                "Show Helper": false,
                "Helper Text": "Helper Text",
                "Show Right Icon": false,
                "Right Icon": "7:29",
                "Show Left Icon": false,
                "Left Icon": "7:29",
                "Label": "Label",
                "Show Text": true,
                "State": "Default",
                "Invalid": "False",
                "Disabled": "False",
                "Filled": "False",
                "Size": "Normal",
                "Ifta Label": "False",
                "Float Label": "False",
                "Float Label Variant": "N/A"
              }
            },
            {
              "type": "INSTANCE",
              "name": "password",
              "componentProperties": {
                "State": "Selected",
                "Toggle Mask": "True",
                "Password Visible": "False"
              }
            },
            {
              "type": "FRAME",
              "name": "Frame 1",
              "layoutMode": "HORIZONTAL",
              "children": [
                {
                  "type": "INSTANCE",
                  "name": "tag",
                  "componentProperties": {
                    "Icon": "7:7029",
                    "Text": "Beliebt",
                    "Show Icon": false,
                    "Severity": "Info",
                    "Rounded": "False"
                  }
                },
                {
                  "type": "INSTANCE",
                  "name": "progressbar",
                  "componentProperties": {
                    "Text": "",
                    "Type": "Basic",
                    "Value": "True"
                  }
                }
              ]
            }
          ]
        },
        {
          "type": "FRAME",
          "name": "footer",
          "layoutMode": "HORIZONTAL",
          "itemSpacing": 7.0,
          "children": [
            {
              "type": "INSTANCE",
              "name": "button",
              "componentProperties": {
                "Left Icon": "7:2160",
                "Icon": "7:2160",
                "Text": "Jetzt starten",
                "Show Right Icon": false,
                "Right Icon": "7:2160",
                "Show Left Icon": false,
                "Severity": "Primary",
                "State": "Idle",
                "Disabled": "False",
                "Icon Only": "False",
                "Raised": "False",
                "Rounded": "False",
                "Text": "False",
                "Outlined": "False",
                "Link": "False"
              }
            }
          ]
        }
      ]
    }
  ]
}
```

```vue
<template>
  <Card
    :pt="{
      root: 'w-md p-8 gap-6',
      body: 'flex flex-col gap-4 !p-0',
      content: 'flex flex-col gap-4',
      footer: 'mt-2',
    }"
  >
    <template #header>
      <h1 class="text-lg font-medium">Anmelden</h1>
    </template>
    <template #content>
      <InputText v-model="email" type="email" placeholder="E-Mail-Adresse" input-id="email-input" />
      <Password v-model="password" input-id="password-input" toggle-mask input-class="w-full" />
      <div class="flex items-center justify-between">
        <Badge value="Beliebt" severity="info" />
        <ProgressBar :value="50" :show-value="false" class="!h-1 w-[84px]" />
      </div>
    </template>
    <template #footer>
      <Button label="Jetzt starten" severity="primary" class="w-full" />
    </template>
  </Card>
</template>

<script setup lang="ts">
  import { ref } from 'vue'
  import Card from 'primevue/card'
  import Badge from 'primevue/badge'
  import Button from 'primevue/button'
  import Password from 'primevue/password'
  import InputText from 'primevue/inputtext'
  import ProgressBar from 'primevue/progressbar'

  const email = ref('')
  const password = ref('password')
</script>
```

Example HARD composite:
```json
{
  "type": "FRAME",
  "name": "Page [Hard composite]",
  "children": [
    {
      "type": "FRAME",
      "name": "datatable",
      "layoutMode": "VERTICAL",
      "children": [
        {
          "type": "FRAME",
          "name": "thead",
          "layoutMode": "HORIZONTAL",
          "children": [
            {
              "type": "TEXT",
              "name": "Projekt",
              "characters": "Projekt"
            },
            {
              "type": "TEXT",
              "name": "Status",
              "characters": "Status"
            },
            {
              "type": "TEXT",
              "name": "Aktionen",
              "characters": "Aktionen"
            }
          ]
        },
        {
          "type": "FRAME",
          "name": "tbody",
          "layoutMode": "VERTICAL",
          "children": [
            {
              "type": "TEXT",
              "name": "Content",
              "characters": "Webseite Relaunch"
            },
            {
              "type": "INSTANCE",
              "name": "tag",
              "componentProperties": {
                "Icon": "7:7029",
                "Text": "Aktiv",
                "Show Icon": false,
                "Severity": "Primary",
                "Rounded": "False"
              }
            },
            {
              "type": "INSTANCE",
              "name": "button",
              "componentProperties": {
                "Right Icon": "7:2160",
                "Left Icon": "7:2160",
                "Show Left Icon": false,
                "Text": "Show",
                "Show Right Icon": false,
                "Icon": "32:3127",
                "Severity": "Primary",
                "State": "Active",
                "Disabled": "False",
                "Icon Only": "True",
                "Raised": "False",
                "Rounded": "False",
                "Text": "True",
                "Outlined": "False",
                "Link": "False"
              }
            }
          ]
        }
      ]
    },
    {
      "type": "FRAME",
      "name": "popover",
      "layoutMode": "VERTICAL",
      "children": [
        {
          "type": "FRAME",
          "name": "popover",
          "layoutMode": "VERTICAL",
          "itemSpacing": 14.0,
          "children": [
            {
              "type": "FRAME",
              "name": "popover",
              "layoutMode": "VERTICAL",
              "children": [
                {
                  "type": "FRAME",
                  "name": "content",
                  "layoutMode": "VERTICAL",
                  "itemSpacing": 7.0,
                  "paddingLeft": 10.5,
                  "paddingRight": 10.5,
                  "paddingTop": 10.5,
                  "paddingBottom": 10.5,
                  "children": [
                    {
                      "type": "FRAME",
                      "name": "col",
                      "layoutMode": "VERTICAL",
                      "itemSpacing": 8.0,
                      "children": [
                        {
                          "type": "INSTANCE",
                          "name": "button",
                          "componentProperties": {
                            "Icon": "7:2160",
                            "Right Icon": "7:2160",
                            "Text": "Bearbeiten",
                            "Left Icon": "32:3139",
                            "Show Right Icon": false,
                            "Show Left Icon": true,
                            "Severity": "Secondary",
                            "State": "Idle",
                            "Disabled": "False",
                            "Icon Only": "False",
                            "Raised": "False",
                            "Rounded": "False",
                            "Text": "False",
                            "Outlined": "True",
                            "Link": "False"
                          }
                        },
                        {
                          "type": "INSTANCE",
                          "name": "button",
                          "componentProperties": {
                            "Icon": "7:2160",
                            "Right Icon": "7:2160",
                            "Text": "Löschen",
                            "Left Icon": "32:3133",
                            "Show Right Icon": false,
                            "Show Left Icon": true,
                            "Severity": "Secondary",
                            "State": "Idle",
                            "Disabled": "False",
                            "Icon Only": "False",
                            "Raised": "False",
                            "Rounded": "False",
                            "Text": "False",
                            "Outlined": "True",
                            "Link": "False"
                          }
                        }
                      ]
                    }
                  ]
                }
              ]
            }
          ]
        }
      ]
    },
    {
      "type": "FRAME",
      "name": "screen",
      "layoutMode": "VERTICAL",
      "paddingTop": 280.0,
      "paddingBottom": 280.0,
      "children": [
        {
          "type": "FRAME",
          "name": "dialog",
          "layoutMode": "VERTICAL",
          "children": [
            {
              "type": "FRAME",
              "name": "header",
              "layoutMode": "HORIZONTAL",
              "paddingLeft": 17.5,
              "paddingRight": 17.5,
              "paddingTop": 17.5,
              "paddingBottom": 17.5,
              "children": [
                {
                  "type": "TEXT",
                  "name": "Projekt bearbeiten",
                  "characters": "Projekt bearbeiten"
                },
                {
                  "type": "FRAME",
                  "name": "actions",
                  "layoutMode": "VERTICAL",
                  "itemSpacing": 7.0,
                  "children": [
                    {
                      "type": "INSTANCE",
                      "name": "button",
                      "componentProperties": {
                        "Text": "Button",
                        "Show Right Icon": true,
                        "Right Icon": "7:2160",
                        "Left Icon": "7:2160",
                        "Show Left Icon": true,
                        "Icon": "9:255",
                        "Severity": "Secondary",
                        "State": "Idle",
                        "Disabled": "False",
                        "Icon Only": "True",
                        "Raised": "False",
                        "Rounded": "False",
                        "Text": "True",
                        "Outlined": "False",
                        "Link": "False"
                      }
                    }
                  ]
                }
              ]
            },
            {
              "type": "FRAME",
              "name": "content",
              "layoutMode": "HORIZONTAL",
              "itemSpacing": 7.0,
              "paddingLeft": 17.5,
              "paddingRight": 17.5,
              "paddingBottom": 17.5,
              "children": [
                {
                  "type": "INSTANCE",
                  "name": "inputtext",
                  "componentProperties": {
                    "Float Label": "Placeholder",
                    "Show Label": true,
                    "Show Helper": false,
                    "Helper Text": "Helper Text",
                    "Show Right Icon": false,
                    "Right Icon": "7:29",
                    "Show Left Icon": false,
                    "Left Icon": "7:29",
                    "Label": "Name",
                    "Show Text": true,
                    "State": "Default",
                    "Invalid": "False",
                    "Disabled": "False",
                    "Filled": "False",
                    "Size": "Normal",
                    "Ifta Label": "False",
                    "Float Label": "False",
                    "Float Label Variant": "N/A"
                  }
                }
              ]
            },
            {
              "type": "FRAME",
              "name": "footer",
              "layoutMode": "HORIZONTAL",
              "itemSpacing": 7.0,
              "paddingLeft": 17.5,
              "paddingRight": 17.5,
              "paddingBottom": 17.5,
              "children": [
                {
                  "type": "INSTANCE",
                  "name": "button",
                  "componentProperties": {
                    "Show Right Icon": false,
                    "Right Icon": "7:2160",
                    "Left Icon": "7:2160",
                    "Text": "Abbrechen",
                    "Icon": "7:2160",
                    "Show Left Icon": false,
                    "Severity": "Secondary",
                    "State": "Idle",
                    "Disabled": "False",
                    "Icon Only": "False",
                    "Raised": "False",
                    "Rounded": "False",
                    "Text": "False",
                    "Outlined": "False",
                    "Link": "False"
                  }
                },
                {
                  "type": "INSTANCE",
                  "name": "button",
                  "componentProperties": {
                    "Left Icon": "7:2160",
                    "Icon": "7:2160",
                    "Text": "Speichern",
                    "Show Right Icon": false,
                    "Right Icon": "7:2160",
                    "Show Left Icon": false,
                    "Severity": "Primary",
                    "State": "Idle",
                    "Disabled": "False",
                    "Icon Only": "False",
                    "Raised": "False",
                    "Rounded": "False",
                    "Text": "False",
                    "Outlined": "False",
                    "Link": "False"
                  }
                }
              ]
            }
          ]
        }
      ]
    }
  ]
}
```

```vue
<template>
  <DataTable :value="projects">
    <Column field="name" header="Name" />
    <Column field="status" header="Status">
      <template #body="{ data }">
        <Tag :value="data.status" :severity="getStatusTagSeverity(data.status)" />
      </template>
    </Column>
    <Column header="Aktionen" header-class="w-24" body-class="w-24 flex justify-center">
      <template #body>
        <Button
          icon="pi pi-ellipsis-h"
          severity="secondary"
          aria-haspopup="true"
          aria-controls="actions-menu"
          @click="actionsMenu?.toggle"
        />
      </template>
    </Column>
  </DataTable>
  <Menu
    ref="actions-menu"
    id="actions-menu"
    :model="actionOptions"
    popup
    :pt="{
      list: 'flex flex-col !gap-2 !p-2.5',
    }"
  >
    <template #item="{ item }">
      <Button
        :label="item.label"
        :icon="item.icon"
        severity="secondary"
        outlined
        class="w-full !justify-start"
      />
    </template>
  </Menu>
  <Dialog
    v-model:visible="isEditProjektDialogVisible"
    header="Projekt bearbeiten"
    modal
    :pt="{
      root: 'w-full max-w-md',
      content: 'flex flex-col !gap-4',
    }"
  >
    <div class="flex flex-col gap-2">
      <label for="name-input" class="text-sm">Name</label>
      <InputText v-model="name" type="text" input-id="name-input" />
    </div>
    <template #footer>
      <Button label="Abbrechen" severity="secondary" />
      <Button label="Speichern" severity="primary" />
    </template>
  </Dialog>
</template>

<script setup lang="ts">
  import { ref, useTemplateRef } from 'vue'
  import Tag from 'primevue/tag'
  import Column from 'primevue/column'
  import DataTable from 'primevue/datatable'
  import Button from 'primevue/button'
  import Menu from 'primevue/menu'
  import Dialog from 'primevue/dialog'
  import InputText from 'primevue/inputtext'

  const projects = ref([
    {
      name: 'Webseite Relaunch',
      status: 'Aktiv',
    },
  ])

  const isEditProjektDialogVisible = ref(true)
  const name = ref('Webseite Relaunch')

  const actionsMenu = useTemplateRef('actions-menu')
  const actionOptions = [
    {
      label: 'Bearbeiten',
      icon: 'pi pi-pen-to-square',
      command: () => (isEditProjektDialogVisible.value = true),
    },
    {
      label: 'Löschen',
      icon: 'pi pi-trash',
    },
  ]

  function getStatusTagSeverity(status: string) {
    switch (status) {
      case 'Aktiv':
        return 'success'
      case 'In Prüfung':
        return 'warn'
      case 'Abgeschlossen':
        return 'info'
      case 'Gestoppt':
        return 'danger'
    }
  }
</script>
```
'''

DOCS_MESSAGE_PROMPT = """PrimeVue documentation for reference:
{context}"""

USER_PROMPT = """Transform the following Figma mockup JSON data into a Vue 3 Single File Component using PrimeVue components.

Figma mockup JSON data:
```json
{figma_json}
```"""

### 3.3 Prompt Builder

In [112]:
def _remove_outer_figma_frame(figma_root: dict) -> dict:
    """If root is a FRAME with one child, unwrap it to reduce nesting noise."""
    if figma_root.get('type') == 'FRAME' and len(figma_root.get('children', [])) == 1:
        return figma_root['children'][0]

    return figma_root


def build_messages(figma_root: dict, strategy: str) -> tuple[list[dict], list[str], int]:
    """Creates the OpenRouter message list with cache breakpoints.

    Structure (static -> variable, prefix-based caching):
      1. System block 1: instructions only        -> Breakpoint 1 (static across ALL calls)
      2. System block 2: few-shot examples        -> Breakpoint 2 (static per prompt strategy)
      3. User block 1:   PrimeVue docs (b2/b3)    -> Breakpoint 3 (repeats per doc combination)
      4. User block 2:   the Figma mockup JSON    -> variable per call, never cached

    cache_control is honored by OpenRouter for providers that need explicit
    caching (Anthropic, Gemini via last breakpoint) and is harmless for
    providers with automatic caching (OpenAI).

    Returns: (messages, used_components, context_tokens)
    """
    figma_root = _remove_outer_figma_frame(figma_root)

    context, used_components = build_context(strategy, figma_root)
    context_tokens = len(context) // 4

     # (1) Instructions only -- no doc context at the end anymore!
    system_content = [{
        'type': 'text',
        'text': SYSTEM_INSTRUCTIONS,
        'cache_control': {'type': 'ephemeral'},          # Breakpoint 1
    }]

    # (2) Few-shot examples as separate static block
    if PROMPT_STRATEGY == 'FEW-SHOT':
        system_content.append({
            'type': 'text',
            'text': FEW_SHOT_EXAMPLES,
            'cache_control': {'type': 'ephemeral'},      # Breakpoint 2
        })

    # (3) + (4) Docs and mockup as two blocks of ONE user message
    #     (avoids consecutive same-role messages being merged/reordered)
    user_content = []

    if context:
        user_content.append({
            'type': 'text',
            'text': DOCS_MESSAGE_PROMPT.format(context=context),
            'cache_control': {'type': 'ephemeral'},      # Breakpoint 3 (b2/b3)
        })

    user_content.append({
        'type': 'text',
        'text': USER_PROMPT.format(
            figma_json=json.dumps(figma_root, ensure_ascii=False, indent=2)
        ),
    })

    messages = [
        {'role': 'system', 'content': system_content},
        {'role': 'user',   'content': user_content},
    ]

    return messages, used_components, context_tokens

## 4. LLM Interaction

In [113]:
def call_llm(messages: list[dict], strategy: str, key: str, model_id: str) -> dict:
    """Calls the OpenRouter Chat Completions API (OpenAI-compatible format,
    works uniformly across OpenAI-, Anthropic- and Google-models).

    model_id: OpenRouter model id, e.g. 'openai/gpt-5.6-sol'

    Returns: {
        'content': str,
        'input_tokens': int,
        'output_tokens': int,
        'cached_tokens': int,       # prompt tokens served from cache
        'cache_discount': float | None,
        'stop_reason': str,
        'duration': float,
        'cost_usd': float | None,   # cost as reported directly by OpenRouter
    }
    """
    metadata = {
        'strategy':   strategy,
        'mockup_key': key,
        'model':      model_id,
    }

    payload = json.dumps( {
        'model': model_id,
        'temperature': API_TEMPERATURE,
        'reasoning': {'effort': API_REASONING_EFFORT},
        'metadata': metadata,
        'usage': {'include': True},
        # Sticky routing: route requests with the same cache prefix to the
        # same provider instance, otherwise Anthropic cache hits are lost.
        'session_id': f'{model_id}/{strategy}/{PROMPT_STRATEGY}',
        'messages': messages,
    }).encode('utf-8')

    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {OPENROUTER_API_KEY}',
    }

    req = urllib.request.Request(
        API_URL,
        data=payload,
        headers=headers,
        method='POST'
    )

    start_time = time.time()

    try:
        with urllib.request.urlopen(req) as resp:
            resp_data = json.loads(resp.read().decode('utf-8'))
    except Exception as e:
        error_text = e.read().decode('utf-8', errors='ignore')

        try:
            error_body = json.loads(error_text)

            detail = error_body.get('error', {}).get('message', error_text)
        except json.JSONDecodeError:
            detail = error_text or str(e)

        raise RuntimeError(f'OpenRouter API Fehler {e.code}: {detail}') from e

    end_time = time.time()

    choice = resp_data['choices'][0]
    usage = resp_data.get('usage', {})

    return {
        'content':        choice['message']['content'],
        'input_tokens':   usage.get('prompt_tokens', 0),
        'output_tokens':  usage.get('completion_tokens', 0),
        'cached_tokens':  usage.get('prompt_tokens_details', {}).get('cached_tokens', 0),
        'cache_discount': usage.get('cache_discount'),
        'stop_reason':    choice.get('finish_reason', ''),
        'duration':       end_time - start_time,
        'cost_usd':       usage.get('cost'),
    }

In [114]:
def extract_sfc(raw: str) -> str:
    """Extracts the Vue SFC code from the LLM response

    Handles three cases:

    1. Direct SFC output (starts with <template>)
    2. Code block with language annotation (```vue ... ```)
    3. Generic code block (``` ... ```)
    """
    # Case 2: ```vue ... ```
    m = re.search(r'```vue\s*\n(.+?)```', raw, re.DOTALL)
    if m:
        return m.group(1).strip()

    # Case 3: ``` ... ```
    m = re.search(r'```\s*\n(.+?)```', raw, re.DOTALL)
    if m:
        candidate = m.group(1).strip()

        if '<template>' in candidate:
            return candidate

    # Case 1: Direct SFC
    if '<template>' in raw:
        start = raw.index('<template>')

        return raw[start:].strip()

    # Fallback: Return the raw response with a comment
    return f'<!-- SFC-Extraktion failed -->\n<!-- RAW:\n{raw[:500]}\n-->'

## 5. Record metrics

In [115]:
_metrics_b: dict = {}

def _reset_metrics_b():
    global _metrics_b
    _metrics_b = {
        'input_tokens':   0,
        'output_tokens':  0,
        'cached_tokens':  0,
        'cache_discount': None,
        'context_tokens': 0,
        'context_components': 0,
        'stop_reason':    '',
        'duration': 0,
        'parse_ok':       False,
        'cost_usd':       None,
    }


def _ast_depth_approx(sfc: str) -> int:
    """Estimates the maximum template depth by counting tags"""
    depth, max_depth = 0, 0
    in_template = False

    for line in sfc.splitlines():
        if '<template>' in line:
            in_template = True

        if not in_template:
            continue

        depth += line.count('<') - line.count('</') - line.count('/>')
        max_depth = max(max_depth, depth)

    return max(0, max_depth)

## 6. Main-Transformation for one Figma JSON with a given strategy

In [116]:
def generate_sfc_b(figma_root: dict, strategy: str, key: str, model_id: str) -> str:
    """Transforms a Figma mockup using the specified LLM strategy and model.

    strategy: 'b1' | 'b2' | 'b3'
    Returns: Vue-3-SFC als String
    """
    _reset_metrics_b()

    messages, used_components, context_tokens = build_messages(figma_root, strategy)

    print(f'Detected Components: {used_components} → Context Tokens: {context_tokens}')

    _metrics_b['context_tokens']     = context_tokens
    _metrics_b['context_components'] = len(used_components)

    response = call_llm(messages, strategy, key, model_id)

    _metrics_b['input_tokens']  = response['input_tokens']
    _metrics_b['output_tokens'] = response['output_tokens']
    _metrics_b['cached_tokens']  = response['cached_tokens']
    _metrics_b['cache_discount'] = response['cache_discount']
    _metrics_b['stop_reason']   = response['stop_reason']
    _metrics_b['duration']      = response['duration']
    _metrics_b['cost_usd']      = response['cost_usd']

    sfc = extract_sfc(response['content'])

    _metrics_b['parse_ok'] = '<template>' in sfc and '<script' in sfc

    return sfc

## 7. Pipeline for all Figma JSONs and all strategies

In [117]:
STRATEGIES = ['b1', 'b2', 'b3']
OUTPUT_PATH = Path(OUTPUT_DIR)

START_MODEL:    str | None = None
START_STRATEGY: str | None = None
START_MOCKUP:   str | None = None

RESULTS_CSV_PATH = Path('reports') / f'results_b_components_{PROMPT_STRATEGY.lower()}.csv'
RESULTS_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

RESULT_FIELDNAMES = [
    'input', 'output', 'complexity', 'model', 'strategy',
    'duration_ms', 'sfc_bytes', 'sfc_lines', 'ast_depth_approx', 'parse_ok',
    'input_tokens', 'cached_tokens', 'cache_discount', 'output_tokens',
    'context_tokens', 'context_components', 'stop_reason', 'cost_usd', 'error',
]

_MODEL_NAMES = list(API_MODELS.keys())


def _load_completed_keys(csv_path: Path) -> set[tuple[str, str, str]]:
    """Reads a previous results CSV (if any) and returns the set of
    (model, strategy, input) combinations that already succeeded.
    Errored attempts are NOT counted as completed, so they get retried."""
    completed: set[tuple[str, str, str]] = set()

    if not csv_path.exists():
        return completed

    with open(csv_path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if row.get('error') in (None, '', 'None'):
                completed.add((row['model'], row['strategy'], row['input']))

    return completed


def _append_result(csv_path: Path, result: dict) -> None:
    """Appends a single result row to the CSV, writing the header once."""
    is_new_file = not csv_path.exists()

    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=RESULT_FIELDNAMES)

        if is_new_file:
            writer.writeheader()

        writer.writerow(result)


def _past_manual_start(model_name: str, strategy: str, key: str) -> bool:
    """True if (model_name, strategy, key) is at or after the manual start point."""
    if START_MODEL is None:
        return True

    mi, smi = _MODEL_NAMES.index(model_name), _MODEL_NAMES.index(START_MODEL)
    if mi != smi:
        return mi > smi

    if START_STRATEGY is None:
        return True

    si, ssi = STRATEGIES.index(strategy), STRATEGIES.index(START_STRATEGY)
    if si != ssi:
        return si > ssi

    if START_MOCKUP is None:
        return True

    figma_keys = list(FIGMA_DATA.keys())

    return figma_keys.index(key) >= figma_keys.index(START_MOCKUP)


def _complexity_from_key(key: str) -> str:
    """Extract complexity prefix from keys like 'medium-8' or 'simple-5'."""
    return key.split('-', 1)[0] if '-' in key else 'unknown'


already_completed = _load_completed_keys(RESULTS_CSV_PATH)
if already_completed:
    print(f'Resuming: {len(already_completed)} combinations already completed in {RESULTS_CSV_PATH}\n')

print(f'Input-Files: {len(FIGMA_DATA)} Strategies: {STRATEGIES}\n')

processed_count = 0

for model_name, model_id in API_MODELS.items():

    print(f'\nUsing model: {model_name} ({model_id})\n{"="*60}')

    for strategy in STRATEGIES:

        print(f' Strategy: {strategy.upper()}')

        for key, figma_root in FIGMA_DATA.items():
            complexity = _complexity_from_key(key)

            if not _past_manual_start(model_name, strategy, key):
                continue

            if (model_name, strategy, key) in already_completed:
                print(f'  SKIP (already done) {key:25s} [{model_name}/{strategy}]')

                continue

            print(f' Processing {key}...\n{"="*60}')

            try:
                sfc = generate_sfc_b(figma_root, strategy, key, model_id)

                out_path = OUTPUT_PATH / complexity / f'{key.split("-", 1)[1]}-{strategy}-{model_name}-{PROMPT_STRATEGY.lower()}.vue'
                out_path.parent.mkdir(parents=True, exist_ok=True)
                out_path.write_text(sfc, encoding='utf-8')

                m = dict(_metrics_b)
                result = {
                    'input':               key,
                    'output':              out_path.name,
                    'complexity':          complexity,
                    'model':               model_name,
                    'strategy':            strategy,
                    'duration_ms':         round(m['duration'] * 1000, 4),
                    'sfc_bytes':           len(sfc),
                    'sfc_lines':           sfc.count('\n') + 1,
                    'ast_depth_approx':    _ast_depth_approx(sfc),
                    'parse_ok':            m['parse_ok'],
                    'input_tokens':        m['input_tokens'],
                    'cached_tokens':       m['cached_tokens'],
                    'cache_discount':      m['cache_discount'],
                    'output_tokens':       m['output_tokens'],
                    'context_tokens':      m['context_tokens'],
                    'context_components':  m['context_components'],
                    'stop_reason':         m['stop_reason'],
                    'cost_usd':            m['cost_usd'],
                    'error':               None,
                }

                print(f'  OK  {key:25s} [{strategy}]  '
                      f'in={m["input_tokens"]:5d}tok  '
                      f'cached={m["cached_tokens"]:5d}tok  '
                      f'out={m["output_tokens"]:4d}tok  '
                      f'${(result["cost_usd"] or 0):.4f}  '
                      f'{m["duration"]:6.0f}ms')

            except Exception as e:
                print(f'  ERROR {key:25s} [{model_name}/{strategy}]  {str(e)}')

                result = {
                    'input': key,
                    'output': None,
                    'complexity': complexity,
                    'model': model_name,
                    'strategy': strategy,
                    'error': str(e),
                    **{k: None for k in [
                        'duration_ms','sfc_bytes','sfc_lines','ast_depth_approx',
                        'parse_ok','input_tokens','output_tokens','cached_tokens',
                        'cache_discount','context_tokens',
                        'context_components','stop_reason','cost_usd'
                    ]},
                }

            _append_result(RESULTS_CSV_PATH, result)

            processed_count += 1


print(f'\nProcessed {processed_count} new transformations this run.')

Input-Files: 30 Strategies: ['b1', 'b2', 'b3']


Using model: claude-sonnet-5 (anthropic/claude-sonnet-5)
 Strategy: B1
 Processing hard-1...
Detected Components: [] → Context Tokens: 0
  OK  hard-1                    [b1]  in=12353tok  cached=    0tok  out= 944tok  $0.0341       9ms
 Processing hard-10...
Detected Components: [] → Context Tokens: 0
  OK  hard-10                   [b1]  in=35390tok  cached=    0tok  out=1313tok  $0.0839      12ms
 Processing hard-2...
Detected Components: [] → Context Tokens: 0
  OK  hard-2                    [b1]  in=19879tok  cached=    0tok  out= 646tok  $0.0462       7ms
 Processing hard-3...
Detected Components: [] → Context Tokens: 0
  OK  hard-3                    [b1]  in=48751tok  cached=    0tok  out= 889tok  $0.1064       8ms
 Processing hard-4...
Detected Components: [] → Context Tokens: 0
  OK  hard-4                    [b1]  in=24701tok  cached=    0tok  out=1088tok  $0.0603      10ms
 Processing hard-5...
Detected Components: [] → Contex

KeyboardInterrupt: 